In [269]:
import sys
import os
import pandas as pd 
import pandas as pd
import re
import csv
import json
from collections import OrderedDict
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [270]:
# Add the project root directory to Python path
sys.path.append("/Users/tahreemyasir/Documents/prelims/DT_hint-1")

from dt_code.llm_response_processing.response_preprocess import process_student_state

In [271]:
input_directory = "/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw"
postState_file = "/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/cleaned_data/matched_postStates.csv"
preState_file = "/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/cleaned_data/preprocess_filtered.csv"

In [272]:

def read_csvs(directory, output_file):
    """
    Reads specific columns from all matching CSV files in the directory and appends them to output_file.
    """
    try:
        # Remove the output file if it exists to avoid appending duplicates
        if os.path.exists(output_file):
            os.remove(output_file)
            
        columns_to_read = [
            "stepPreState",
            "stepPostState",
            "currentProblem",
            "currentProblemType",
            "currentProblemDescription",
            "currentProblemMetaData",
        ]
        
        first_write = not os.path.exists(output_file)
        # list_of_files = ['23S']
        list_of_files = ['23F', '23S', '25S', '24F']
        for file in list_of_files:
            for i in range(1, 7):
                filename = f"actionLog_L7_{i}_{file}.csv"
                filename = os.path.join(directory, filename)
                print("Filename: ", filename)
                if os.path.exists(filename):
                    df = pd.read_csv(filename, usecols=columns_to_read, low_memory=False)
                    df.to_csv(output_file, mode='a', header=first_write, index=False)
                    first_write = False  # Only write header for the first file
                else:
                    print(f"File not found: {filename}")
            
        
    except Exception as e:
        print(f"Error in preprocess_step_student_model_csvs: {str(e)}")
        raise
    
# Step 1: Preprocess and collect relevant columns
read_csvs(input_directory, postState_file)


Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_1_23F.csv
Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_2_23F.csv
Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_3_23F.csv
Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_4_23F.csv
Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_5_23F.csv
Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_6_23F.csv
Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_1_23S.csv
Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_2_23S.csv
Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_3_23S.csv
Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_4_23S.csv
Filename:  /Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/Raw/actionLog_L7_5_23S.csv

In [273]:
import pandas as pd 
post_df = pd.read_csv('/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/cleaned_data/matched_postStates.csv')
print("Total number of instances: ",post_df.shape)
post_df.head()
# drop null values from post_df
post_df = post_df.dropna(subset=['stepPreState','stepPostState'])
post_df.shape

Total number of instances:  (3671035, 6)


(3538746, 6)

In [274]:
pre_df = pd.read_csv(preState_file)
print("Total number of instances: ",pre_df.shape)
pre_df.head()
pre_df['stepPreState'].nunique()

Total number of instances:  (558, 10)


558

In [275]:

# drop same prestates and poststates in post_df
post_df.drop(index=post_df[post_df['stepPreState'] == post_df['stepPostState']].index, inplace=True)
print("removed equal pre post states in post_df: ", post_df.shape)
# only keep rows where post_df['currentProblemType'] == 'PS'
post_df = post_df[post_df['currentProblemType'] == 'PS']
print("extracted rows where post_df['currentProblemType'] == 'PS': ", post_df.shape)
# extract all rows of post_df where pre_df['stepPreState'] == post_df['stepPreState']
post_df = post_df[post_df['stepPreState'].isin(pre_df['stepPreState'])]
print("extracted rows where pre_df[prestates] == post_df[prestates]: ", post_df.shape)
# add whatever is after / in post_df['stepPreState'] to post_df['conclusion']
post_df['conclusion'] = post_df['stepPreState'].str.split('/').str[1]
#remove everything inlcluding / and after / in post_df['stepPostState']
post_df['stepPreState'] = post_df['stepPreState'].astype(str).str.replace(r'/.*$', '', regex=True)
post_df['stepPostState'] = post_df['stepPostState'].astype(str).str.replace(r'/.*$', '', regex=True)
post_df[['stepPreState','stepPostState']].head(10)
print("unique pre states in post_df: ", post_df['stepPreState'].nunique())
post_df.head()

removed equal pre post states in post_df:  (515753, 6)
extracted rows where post_df['currentProblemType'] == 'PS':  (468782, 6)
extracted rows where pre_df[prestates] == post_df[prestates]:  (25650, 6)
unique pre states in post_df:  558


,stepPreState,stepPostState,currentProblem,currentProblemType,currentProblemDescription,currentProblemMetaData,conclusion
282,"(L*K)[1;0;Given],((K*O)>(Q>P))[2;0;Given],O[3;0;Given],(P>R)[4;0;Given]","(L*K)[1;0;Given],((K*O)>(Q>P))[2;0;Given],O[3;0;Given],(P>R)[4;0;Given],((O*K)>(Q>P))[5;2;Commutative]",2.5,PS,"L*K,(K*O)>(Q>P),O,P>R/Q>R","SIMP,CONJ,MP,HS",(Q>R)
427,"((S>D)+I)[1;0;Given],((-S+Q)>Y)[2;0;Given],-D[3;0;Given],(-D>-I)[4;0;Given]","((S>D)+I)[1;0;Given],((-S+Q)>Y)[2;0;Given],-D[3;0;Given],(-D>-I)[4;0;Given],-I[5;3,4;Modus Ponens]",3.4,PS,"(S>D)+I,(-S+Q)>Y,-D,-D>-I/Y","MP,DS,MT,ADD",Y
558,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given]","(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma]",4.4,PS,"S+B,B>D,S>G/D+G","CD,MP,COMM",(D+G)
564,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma]","(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative]",4.4,PS,"S+B,B>D,S>G/D+G","CD,MP,COMM",(D+G)
568,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative]","(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative],(D+G)[6;5,4;Modus Ponens]",4.4,PS,"S+B,B>D,S>G/D+G","CD,MP,COMM",(D+G)


In [276]:
# Function to remove pre-state elements from post-state
def remove_pre_from_post(pre_state, post_state):
    pre_items = [x.strip() for x in pre_state.split(",")]
    post_items = [x.strip() for x in post_state.split(",")]
    
    # keep only new items that are not already in pre-state
    new_items = [item for item in post_items if item not in pre_items]
    
    return ",".join(new_items)

# Apply the function
post_df["stepPostState_processed"] = post_df.apply(lambda row: remove_pre_from_post(row["stepPreState"], row["stepPostState"]), axis=1)
print("unique pre states in post_df: ", post_df['stepPreState'].nunique())
post_df[['stepPreState','stepPostState','stepPostState_processed']].head(10)


unique pre states in post_df:  558


,stepPreState,stepPostState,stepPostState_processed
282,"(L*K)[1;0;Given],((K*O)>(Q>P))[2;0;Given],O[3;0;Given],(P>R)[4;0;Given]","(L*K)[1;0;Given],((K*O)>(Q>P))[2;0;Given],O[3;0;Given],(P>R)[4;0;Given],((O*K)>(Q>P))[5;2;Commutative]",((O*K)>(Q>P))[5;2;Commutative]
427,"((S>D)+I)[1;0;Given],((-S+Q)>Y)[2;0;Given],-D[3;0;Given],(-D>-I)[4;0;Given]","((S>D)+I)[1;0;Given],((-S+Q)>Y)[2;0;Given],-D[3;0;Given],(-D>-I)[4;0;Given],-I[5;3,4;Modus Ponens]","-I[5;3,4;Modus Ponens]"
558,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given]","(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma]","((B+S)>(D+G))[4;2,3;Constructive Dilemma]"
564,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma]","(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative]",(B+S)[5;1;Commutative]
568,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative]","(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative],(D+G)[6;5,4;Modus Ponens]","(D+G)[6;5,4;Modus Ponens]"
577,"(Z>(-Y>X))[1;0;Given],(Z*-W)[2;0;Given],(W+(T>S))[3;0;Given],(-Y+T)[4;0;Given]","(Z>(-Y>X))[1;0;Given],(Z*-W)[2;0;Given],(W+(T>S))[3;0;Given],(-Y+T)[4;0;Given],-W[5;2;Simplification]",-W[5;2;Simplification]
582,"(Z>(-Y>X))[1;0;Given],(Z*-W)[2;0;Given],(W+(T>S))[3;0;Given],(-Y+T)[4;0;Given],-W[5;2;Simplification]","(Z>(-Y>X))[1;0;Given],(Z*-W)[2;0;Given],(W+(T>S))[3;0;Given],(-Y+T)[4;0;Given],-W[5;2;Simplification],(T>S)[6;5,3;Disjunctive Syllogism]","(T>S)[6;5,3;Disjunctive Syllogism]"
1994,"-(G*A)[1;0;Given],(B>A)[2;0;Given]","-(G*A)[1;0;Given],(B>A)[2;0;Given],(-G+-A)[3;1;DeMorgan's Law]",(-G+-A)[3;1;DeMorgan's Law]
1998,"-(G*A)[1;0;Given],(B>A)[2;0;Given],(-G+-A)[3;1;DeMorgan's Law]","-(G*A)[1;0;Given],(B>A)[2;0;Given],(-G+-A)[3;1;DeMorgan's Law],(G>-A)[4;3;Conditional Identity (Implication)]",(G>-A)[4;3;Conditional Identity (Implication)]
2282,"-(-G*B)[1;0;Given],(G>D)[2;0;Given]","-(-G*B)[1;0;Given],(G>D)[2;0;Given],(--G+-B)[3;1;DeMorgan's Law]",(--G+-B)[3;1;DeMorgan's Law]


In [277]:
# extract the next step from post state 
def split_first_top_level_comma(s):
    # Split on first comma that is NOT inside (), [] or {}
    parts = re.split(r',(?![^()\[\]]*[\]\)])', s, maxsplit=1)
    return parts[0].strip()

post_df["next_step_rule"] = post_df["stepPostState_processed"].apply(split_first_top_level_comma)
# extrcat everything before the first '['
post_df["next_step"] = post_df["stepPostState_processed"].str.split("[", n=1).str[0].str.strip()
# Extract text between the last ';' and the closing ']'
post_df["rule"] = post_df["stepPostState_processed"].str.extract(r";([^;\]]+)\]")
print("unique pre states in post_df: ", post_df['stepPreState'].nunique())
print("size of post_df: ", post_df.shape)
post_df[['stepPostState','stepPostState_processed', 'next_step_rule', 'next_step', 'rule']].head(10)



unique pre states in post_df:  558
size of post_df:  (25650, 11)


,stepPostState,stepPostState_processed,next_step_rule,next_step,rule
282,"(L*K)[1;0;Given],((K*O)>(Q>P))[2;0;Given],O[3;0;Given],(P>R)[4;0;Given],((O*K)>(Q>P))[5;2;Commutative]",((O*K)>(Q>P))[5;2;Commutative],((O*K)>(Q>P))[5;2;Commutative],((O*K)>(Q>P)),Commutative
427,"((S>D)+I)[1;0;Given],((-S+Q)>Y)[2;0;Given],-D[3;0;Given],(-D>-I)[4;0;Given],-I[5;3,4;Modus Ponens]","-I[5;3,4;Modus Ponens]","-I[5;3,4;Modus Ponens]",-I,Modus Ponens
558,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma]","((B+S)>(D+G))[4;2,3;Constructive Dilemma]","((B+S)>(D+G))[4;2,3;Constructive Dilemma]",((B+S)>(D+G)),Constructive Dilemma
564,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative]",(B+S)[5;1;Commutative],(B+S)[5;1;Commutative],(B+S),Commutative
568,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative],(D+G)[6;5,4;Modus Ponens]","(D+G)[6;5,4;Modus Ponens]","(D+G)[6;5,4;Modus Ponens]",(D+G),Modus Ponens
577,"(Z>(-Y>X))[1;0;Given],(Z*-W)[2;0;Given],(W+(T>S))[3;0;Given],(-Y+T)[4;0;Given],-W[5;2;Simplification]",-W[5;2;Simplification],-W[5;2;Simplification],-W,Simplification
582,"(Z>(-Y>X))[1;0;Given],(Z*-W)[2;0;Given],(W+(T>S))[3;0;Given],(-Y+T)[4;0;Given],-W[5;2;Simplification],(T>S)[6;5,3;Disjunctive Syllogism]","(T>S)[6;5,3;Disjunctive Syllogism]","(T>S)[6;5,3;Disjunctive Syllogism]",(T>S),Disjunctive Syllogism
1994,"-(G*A)[1;0;Given],(B>A)[2;0;Given],(-G+-A)[3;1;DeMorgan's Law]",(-G+-A)[3;1;DeMorgan's Law],(-G+-A)[3;1;DeMorgan's Law],(-G+-A),DeMorgan's Law
1998,"-(G*A)[1;0;Given],(B>A)[2;0;Given],(-G+-A)[3;1;DeMorgan's Law],(G>-A)[4;3;Conditional Identity (Implication)]",(G>-A)[4;3;Conditional Identity (Implication)],(G>-A)[4;3;Conditional Identity (Implication)],(G>-A),Conditional Identity (Implication)
2282,"-(-G*B)[1;0;Given],(G>D)[2;0;Given],(--G+-B)[3;1;DeMorgan's Law]",(--G+-B)[3;1;DeMorgan's Law],(--G+-B)[3;1;DeMorgan's Law],(--G+-B),DeMorgan's Law


In [278]:
# Helper: unique items preserving order
def unique_in_order(seq):
    return list(OrderedDict.fromkeys(seq))

# Group and aggregate
grouped_df = (
    post_df.groupby('stepPreState', sort=False, as_index=False)
    .agg({
        'next_step': lambda x: unique_in_order(list(x)),
        'rule': lambda x: unique_in_order(list(x))
    })
    .rename(columns={'next_step': 'step_list', 'rule': 'rule_list'})
)

# Merge back with original DataFrame
final_df = post_df.merge(grouped_df, on='stepPreState', how='left')
# Keep only the first occurrence of each stepPreState
final_df_unique = final_df.drop_duplicates(subset='stepPreState', keep='first')
print("final_df_unique: ", final_df_unique.shape)
print("final_df unique prestates: ", final_df_unique['stepPreState'].nunique())
final_df_unique.head()

final_df_unique:  (558, 13)
final_df unique prestates:  558


,stepPreState,stepPostState,currentProblem,currentProblemType,currentProblemDescription,currentProblemMetaData,conclusion,stepPostState_processed,next_step_rule,next_step,rule,step_list,rule_list
0,"(L*K)[1;0;Given],((K*O)>(Q>P))[2;0;Given],O[3;0;Given],(P>R)[4;0;Given]","(L*K)[1;0;Given],((K*O)>(Q>P))[2;0;Given],O[3;0;Given],(P>R)[4;0;Given],((O*K)>(Q>P))[5;2;Commutative]",2.5,PS,"L*K,(K*O)>(Q>P),O,P>R/Q>R","SIMP,CONJ,MP,HS",(Q>R),((O*K)>(Q>P))[5;2;Commutative],((O*K)>(Q>P))[5;2;Commutative],((O*K)>(Q>P)),Commutative,"[((O*K)>(Q>P)), K, (-(K*O)+(Q>P)), ((P>R)*((K*O)>(Q>P))), (((K*O)>(Q>P))+R), ((L*K)+Q), (-P+R), (O+K), (-R>-P), (((K*O)+P)>((Q>P)+R)), (((K*O)>(Q>P))*(P>R)), ((L*K)*O), ((K*O)>(-Q+P)), L, ((P+(K*O))>(R+(Q>P))), (--L*K), (O*(L*K)), ((P>R)*O), (O+-K), ((K*O)>(-P>-Q)), (-(Q>P)>-(K*O)), (Q>R), --O, (O+-P), ((-O+L)>(M*-N)), (K*L), (B>(A>J)), (K+O), (O+P), ((F+G)>H), (((K*O)*P)>((Q>P)*R)), (O*((K*O)>(Q>P))), ((L*K)*(L*K))]","[Commutative, Simplification, Conditional Identity (Implication), Conjunction, Addition, Contrapositive, Constructive Dilemma, Double Negation, Hypothetical Syllogism, Given, Implication]"
1,"((S>D)+I)[1;0;Given],((-S+Q)>Y)[2;0;Given],-D[3;0;Given],(-D>-I)[4;0;Given]","((S>D)+I)[1;0;Given],((-S+Q)>Y)[2;0;Given],-D[3;0;Given],(-D>-I)[4;0;Given],-I[5;3,4;Modus Ponens]",3.4,PS,"(S>D)+I,(-S+Q)>Y,-D,-D>-I/Y","MP,DS,MT,ADD",Y,"-I[5;3,4;Modus Ponens]","-I[5;3,4;Modus Ponens]",-I,Modus Ponens,"[-I, (((-S+Q)+-D)>(Y+-I)), (I>D), ((-S+D)+I), (D+-I), (-(-S+Q)+Y), (((-S+Q)>Y)+S), (-(S>D)>I), (--D+-I), (-Y>-(-S+Q)), (-D+Q), ((S>Q)>Y), (I+(S>D)), ((--S>Q)>Y), ((-D>-S)+I), (-D+Y), (-D+I), (-D+-I), ((Q+-S)>Y), (-D+S), ---D, (-D+-S), ((-D>-I)*-D), (((S>D)+I)+S), (C*-F), (-D*(-D>-I)), Y, (Z>K), (--I>--D), --((-S+Q)>Y)]","[Modus Ponens, Constructive Dilemma, Contrapositive, Conditional Identity (Implication), Addition, Commutative, Double Negation, Conjunction, Given, Implication]"
2,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given]","(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma]",4.4,PS,"S+B,B>D,S>G/D+G","CD,MP,COMM",(D+G),"((B+S)>(D+G))[4;2,3;Constructive Dilemma]","((B+S)>(D+G))[4;2,3;Constructive Dilemma]",((B+S)>(D+G)),Constructive Dilemma,"[((B+S)>(D+G)), (-S+G), (-S>B), (B+S), ((B>D)*(S>G)), (-B+D), ((S+B)>(G+D)), (-G>-S), (-D>-B), ((S+B)*(S>G)), (D+G), ((B>D)+S), ((B>D)*(S+B)), ((S>G)*(S+B)), ((S+B)*(B>D)), ((A>B)*(-D>F)), -(D+G), (-G*K), ((S>G)+D), ((S*B)>(G*D)), ((S+B)+S), ((S>G)*(B>D)), ((B*S)>(D*G)), ((S>G)+B), (--S+B), ((S>G)+G), --(S+B)]","[Constructive Dilemma, Conditional Identity (Implication), Commutative, Conjunction, Contrapositive, Addition, Implication, Given, Negation of Conclusion, Double Negation]"
3,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma]","(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative]",4.4,PS,"S+B,B>D,S>G/D+G","CD,MP,COMM",(D+G),(B+S)[5;1;Commutative],(B+S)[5;1;Commutative],(B+S),Commutative,"[(B+S), (-(D+G)>-(B+S)), (-S>B), , ((B+S)>(-D>G)), (-(B+S)+(D+G)), (((B+S)+B)>((D+G)+D)), (((B+S)>(D+G))*(S>G)), ((S+B)>(D+G)), ((B+S)>(G+D)), (-G>-S), (D+G), ((S+B)*((B+S)>(D+G))), (((B+S)>(D+G))*(S+B)), (((B+S)>(D+G))+S), ((-B>S)>(D+G)), ((S+B)*(B>D))]","[Commutative, Contrapositive, Conditional Identity (Implication), nan, Constructive Dilemma, Conjunction, Modus Ponens, Implication, Addition]"
4,"(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative]","(S+B)[1;0;Given],(B>D)[2;0;Given],(S>G)[3;0;Given],((B+S)>(D+G))[4;2,3;Constructive Dilemma],(B+S)[5;1;Commutative],(D+G)[6;5,4;Modus Ponens]",4.4,PS,"S+B,B>D,S>G/D+G","CD,MP,COMM",(D+G),"(D+G)[6;5,4;Modus Ponens]","(D+G)[6;5,4;Modus Ponens]",(D+G),Modus Ponens,"[(D+G), , (-B>S), (-(D+G)>-(B+S)), (--B+S), ((B+S)*(B>D))]","[Modus Ponens, nan, Conditional Identity (Implication), Contrapositive, Double Negation, Conjunction

In [279]:
# sort df based on currentProblem
final_df_unique = final_df_unique.sort_values('currentProblem')
# Write to CSV
student_step = "/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/cleaned_data/actual_student_steps.csv"
final_df_unique.to_csv(student_step, index=False)


In [280]:
final_df_unique.columns



Index(['stepPreState', 'stepPostState', 'currentProblem', 'currentProblemType',
       'currentProblemDescription', 'currentProblemMetaData', 'conclusion',
       'stepPostState_processed', 'next_step_rule', 'next_step', 'rule',
       'step_list', 'rule_list'],
      dtype='object')

In [281]:
def process_givens(obj):
    """
    Process the givens from the object.
    Returns a list of givens and a list of processed givens lines.
    """
    # Access 'Givens' key (capitalized)
    givens = obj.get('Givens', [])
    
    # Ensure it is a list of strings
    if isinstance(givens, str):
        # Fallback: split comma-separated string
        givens = [g.strip() for g in givens.split(',') if g.strip()]
    
    processed_givens_lines = [f"  {idx}. {given}" for idx, given in enumerate(givens, 1)]
    
    return givens, processed_givens_lines

def process_intermediates(obj, givens):
    """
    Process the intermediates from the object.
    Returns a list of intermediates and a list of processed intermediates lines.
    """
    intermediates = obj.get('Intermediates', {})  # Capitalized key
    expressions = intermediates.get('Expressions', [])
    rules = intermediates.get('Rules', [])
    
    start_idx = len(givens) + 1
    processed_intermediates_lines = []

    for idx, (expr, rule) in enumerate(zip(expressions, rules), start=start_idx):
        rule_name = rule
        # Handle rules with semicolons
        if isinstance(rule, str) and ';' in rule:
            parts = rule.strip('[]').split(';')
            # Remove numeric parts equal to the current index
            refs = [p for p in parts[:-1] if not (p.isdigit() and int(p) == idx)]
            rule_name = ';'.join(refs + [parts[-1]]) if refs else parts[-1]
        
        line = f"  {idx}. {expr}  [{rule_name}]"
        processed_intermediates_lines.append(line)
    
    next_idx = start_idx + len(processed_intermediates_lines)
    return processed_intermediates_lines, next_idx

In [282]:
def parse_sPreState(sPreState):
    if '/' in sPreState:
        rule_part, conclusion = sPreState.rsplit('/', 1)
        #print("conclusion: ", conclusion)
    else:
        rule_part = sPreState
        conclusion = ""

    # Split by commas but keep parentheses and brackets together
    entries = [e.strip() for e in re.split(r'(?<=\])', rule_part) if e.strip()]
    #print("entries: ", entries)
    givens = []
    expressions = []
    rules = []

    for entry in entries:
        entry = entry.lstrip(',')
        #print("entry: ", entry)
        match = re.split(r'(?=\[)', entry)
        #print("match: ", match)
        if "Given" in match[1]:
            givens.append(match[0])
        else: 
            #print("match: ", match)
            expressions.append(match[0])
            rules.append(match[1])
    return {
         "Givens": givens,
         "Intermediates": {
             "Expressions": expressions,
             "Rules": rules
         },
    }

def convert_csv_to_jsonl(csv_file_path, jsonl_file_path):
    with open(csv_file_path, newline='', encoding='utf-8') as csvfile, \
         open(jsonl_file_path, 'w', encoding='utf-8') as jsonlfile:

        reader = csv.DictReader(csvfile)
        for row in reader:
            # Parse the pre-state
            spre = parse_sPreState(row["stepPreState"])
            
            # Process givens
            givens, processed_givens_lines = process_givens(spre)
            
            # Process intermediates
            processed_intermediates_lines, next_idx = process_intermediates(spre, givens)
            
            # Build entry to write
            entry = {
                "currentProblem": row.get("currentProblem", ""),
                "conclusion": row.get("conclusion", ""),
                "givens": processed_givens_lines,
                "intermediates": processed_intermediates_lines,
                "step_list": row.get("step_list", ""),
                "rule_list": row.get("rule_list", ""),
            }
            print("entry: ", entry)

            # Write as JSONL (one JSON object per line)
            jsonlfile.write(json.dumps(entry, ensure_ascii=False) + '\n')

In [283]:
import json

def remove_rows_where_conclusion_in_intermediates(jsonl_file_path):
    """
    Remove rows where the conclusion appears as a separate expression
    in the intermediates list (exact match, not substring).
    """
    with open(jsonl_file_path, 'r') as f:
        lines = f.readlines()
    
    filtered_lines = []
    removed_count = 0

    for line in lines:
        try:
            entry = json.loads(line.strip())
            conclusion = entry.get('conclusion', '').strip()
            intermediates = [expr.strip() for expr in entry.get('intermediates', [])]

            if conclusion not in intermediates:
                filtered_lines.append(line)
            else:
                removed_count += 1

        except json.JSONDecodeError:
            # Keep lines that can't be parsed
            filtered_lines.append(line)

    # Write filtered lines back to file
    with open(jsonl_file_path, 'w') as f:
        f.writelines(filtered_lines)

    print(f"Removed {removed_count} rows where conclusion is a separate expression")
    print(f"Remaining rows: {len(filtered_lines)}")


In [284]:
def show_pivot_table_by_problem(jsonl_file_path):
    """
    Show pivot table based on currentProblem
    """
    # Read the JSONL file and extract data
    data = []
    
    with open(jsonl_file_path, 'r') as f:
        for line in f:
            try:
                entry = json.loads(line.strip())
                data.append({
                    'currentProblem': entry.get('currentProblem', ''),
                    'sAssertion': entry.get('sAssertion', ''),
                    'problemDescription': entry.get('currentProblemDescription', '')
                })
            except json.JSONDecodeError:
                continue
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Create pivot table
    pivot_table = df.groupby('currentProblem').agg({
        'sAssertion': 'count'
    }).reset_index()
    
    pivot_table.columns = ['Problem_Number', 'Instance_Count']
    
    # Sort by problem number
    pivot_table = pivot_table.sort_values('Problem_Number')
    
    # print("Pivot Table - Problem Numbers and Instance Counts:")
    # print(pivot_table.to_string(index=False))
    
    # Show summary
    print(f"\nTotal problems: {len(pivot_table)}")
    print(f"Total instances: {pivot_table['Instance_Count'].sum()}")
    
    return pivot_table

In [285]:

csv_file = "/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/cleaned_data/actual_student_steps.csv"     # Update if needed
jsonl_file = "/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/cleaned_data/actual_student_steps.jsonl"
convert_csv_to_jsonl(csv_file, jsonl_file)
remove_rows_where_conclusion_in_expressions(jsonl_file)

# Add this line to print the pivot table
# show_pivot_table_by_problem(jsonl_file)

entry:  {'currentProblem': '2.2', 'conclusion': 'H', 'givens': ['  1. ((F+G)>H)', '  2. (I+F)', '  3. (-I*J)'], 'intermediates': ['  4. -I  [3;Simplification]', '  5. F  [2,4;Disjunctive Syllogism]'], 'step_list': "['(F+G)', '(-H>-(F+G))', '((G+F)>H)', 'J', '(-(F+G)+H)', '(F+-G)', '((-F>G)>H)']", 'rule_list': "['Addition', 'Contrapositive', 'Commutative', 'Simplification', 'Conditional Identity (Implication)']"}
entry:  {'currentProblem': '2.2', 'conclusion': 'H', 'givens': ['  1. ((F+G)>H)', '  2. (I+F)', '  3. (-I*J)'], 'intermediates': [], 'step_list': "['((I+F)*(-I*J))', '(B>(A>J))', '-H', '-I', '(A>C)', '(-I>F)', '(F+I)', '((I+F)*((F+G)>H))', '((-F>G)>H)', '((-I*J)*(I+F))']", 'rule_list': "['Conjunction', 'Given', 'Negation of Conclusion', 'Simplification', 'Conditional Identity (Implication)', 'Commutative']"}
entry:  {'currentProblem': '2.3', 'conclusion': 'N', 'givens': ['  1. ((-K+L)>(M*N))', '  2. (K>O)', '  3. -O'], 'intermediates': ['  4. -K  [3,2;Modus Tollens]', '  5. (-K